# 🎬 Movie Recommendation AI Assistant with RAG

**Project:** Mid-term AI Knowledge Assistant

**Features:**
- RAG with FAISS for semantic movie search
- LangChain tools for watchlist management
- Conversation memory with summarization
- Interactive validation
- Performance monitoring
- Graceful degradation

**Model:** Groq `openai/gpt-oss-120b`

---

## 1. Setup & Installation 📦

In [1]:
# Import libraries
import os
import json
import time
import ast
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import Tool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic.memory.buffer import ConversationBufferMemory
from langchain_groq import ChatGroq
from langchain_classic import hub

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [2]:
# Create project structure
os.makedirs('data', exist_ok=True)
os.makedirs('memory', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("✅ Project folders created!")

✅ Project folders created!


## 2. Configuration & Setup ⚙️

In [3]:
# Configuration
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
MODEL_NAME = 'openai/gpt-oss-120b'

# Paths
MOVIES_CSV = 'data/movies_metadata.csv'
CREDITS_CSV = 'data/credits.csv'
WATCHLIST_FILE = 'my_watchlist.xlsx'
MEMORY_FILE = 'memory/conversation_memory.json'
BACKUP_FILE = 'memory/full_backup.json'
LOG_FILE = 'logs/session.log'

# OPTIMIZED Memory settings
MAX_RECENT_MESSAGES = 4  # Reduced from 10
# MAX_SUMMARY_WORDS = 300  # Reduced from 500
# SUMMARIZE_EVERY = 8      # More frequent

# Verify API key
if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY not found in .env file!")

print("✅ Configuration loaded successfully!")
print(f"📍 Movies CSV: {MOVIES_CSV}")
print(f"📍 Watchlist: {WATCHLIST_FILE}")

✅ Configuration loaded successfully!
📍 Movies CSV: data/movies_metadata.csv
📍 Watchlist: my_watchlist.xlsx


## 3. Data Loading & Cleaning 🧹

In [4]:
# Load movies metadata
print("📥 Loading movies_metadata.csv...")
movies_df = pd.read_csv(MOVIES_CSV, low_memory=False)
print(f"   Loaded {len(movies_df)} rows")

# Load credits
print("📥 Loading credits.csv...")
credits_df = pd.read_csv(CREDITS_CSV)
print(f"   Loaded {len(credits_df)} rows")

print("\n✅ Raw data loaded!")

📥 Loading movies_metadata.csv...
   Loaded 45466 rows
📥 Loading credits.csv...
   Loaded 45476 rows

✅ Raw data loaded!


In [5]:
print(movies_df.head())
print(movies_df.columns.tolist())

   adult                              belongs_to_collection    budget  \
0  False  {'id': 10194, 'name': 'Toy Story Collection', ...  30000000   
1  False                                                NaN  65000000   
2  False  {'id': 119050, 'name': 'Grumpy Old Men Collect...         0   
3  False                                                NaN  16000000   
4  False  {'id': 96871, 'name': 'Father of the Bride Col...         0   

                                              genres  \
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   
3  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
4                     [{'id': 35, 'name': 'Comedy'}]   

                               homepage     id    imdb_id original_language  \
0  http://toystory.disney.com/toy-story    862  tt0114709                en   
1                                   NaN   8844  tt0113497         

In [6]:
print(credits_df.head())
print(credits_df.columns.tolist())

                                                cast  \
0  [{'cast_id': 14, 'character': 'Woody (voice)',...   
1  [{'cast_id': 1, 'character': 'Alan Parrish', '...   
2  [{'cast_id': 2, 'character': 'Max Goldman', 'c...   
3  [{'cast_id': 1, 'character': "Savannah 'Vannah...   
4  [{'cast_id': 1, 'character': 'George Banks', '...   

                                                crew     id  
0  [{'credit_id': '52fe4284c3a36847f8024f49', 'de...    862  
1  [{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...   8844  
2  [{'credit_id': '52fe466a9251416c75077a89', 'de...  15602  
3  [{'credit_id': '52fe44779251416c91011acb', 'de...  31357  
4  [{'credit_id': '52fe44959251416c75039ed7', 'de...  11862  
['cast', 'crew', 'id']


In [7]:
print(movies_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  str    
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [5]:
# Data cleaning function
def clean_movies_data(df):
    """Clean and prepare movie data"""
    print("🧹 Cleaning data...")
    
    # Keep only essential columns
    columns_to_keep = ['id', 'title', 'overview', 'genres', 'release_date', 
                    'runtime', 'vote_average', 'vote_count', 'popularity']
    df = df[columns_to_keep]
    
    # Remove rows without plots (critical for RAG!)
    before = len(df)
    df = df.dropna(subset=['overview', 'title'])
    print(f"   Removed {before - len(df)} rows without plot/title")
    
    # Remove rows with empty overview
    df = df[df['overview'].str.strip().str.len() > 10]
    
    # Parse genres (JSON string to list)
    def parse_genres(x):
        try:
            genres = ast.literal_eval(x)
            return [g['name'] for g in genres]
        except:
            return []
    
    df['genres'] = df['genres'].apply(parse_genres)
    df['genre_names'] = df['genres'].apply(lambda x: ', '.join(x) if x else 'Unknown')
    
    # Extract year from release_date
    df['year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
    
    # Clean runtime
    df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce')
    
    # Clean ratings
    df['vote_average'] = pd.to_numeric(df['vote_average'], errors='coerce')
    df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce')
    
    # Remove movies with missing critical info
    df = df.dropna(subset=['year', 'vote_average'])
    
    # Reset index
    df = df.reset_index(drop=True)
    
    print(f"✅ Cleaned! Final dataset: {len(df)} movies")
    return df

movies_df = clean_movies_data(movies_df)

🧹 Cleaning data...
   Removed 960 rows without plot/title
✅ Cleaned! Final dataset: 44427 movies


In [6]:
# Parse credits data
def parse_credits(df):
    """Extract cast and crew from credits"""
    print("🎭 Processing credits...")
    
    def get_director(crew_str):
        try:
            crew = ast.literal_eval(crew_str)
            directors = [c['name'] for c in crew if c['job'] == 'Director']
            return directors[0] if directors else 'Unknown'
        except:
            return 'Unknown'
    
    def get_cast(cast_str, n=5):
        try:
            cast = ast.literal_eval(cast_str)
            return [c['name'] for c in cast[:n]]
        except:
            return []
    
    df['director'] = df['crew'].apply(get_director)
    df['cast_list'] = df['cast'].apply(get_cast)
    df['cast_names'] = df['cast_list'].apply(lambda x: ', '.join(x) if x else 'Unknown')
    
    # Keep only needed columns
    df = df[['id', 'director', 'cast_names', 'cast_list']]
    
    print(f"✅ Credits processed!")
    return df

credits_df = parse_credits(credits_df)

🎭 Processing credits...
✅ Credits processed!


In [7]:
# Merge movies with credits
movies_df['id'] = pd.to_numeric(movies_df['id'], errors='coerce')
credits_df['id'] = pd.to_numeric(credits_df['id'], errors='coerce')

movies_df = movies_df.merge(credits_df, on='id', how='left')

# Fill missing values
movies_df['director'] = movies_df['director'].fillna('Unknown')
movies_df['cast_names'] = movies_df['cast_names'].fillna('Unknown')

print(f"✅ Final dataset ready: {len(movies_df)} movies with complete info")
print(f"\n📊 Sample:")
print(movies_df.head())

✅ Final dataset ready: 44503 movies with complete info

📊 Sample:
      id                        title  \
0    862                    Toy Story   
1   8844                      Jumanji   
2  15602             Grumpier Old Men   
3  31357            Waiting to Exhale   
4  11862  Father of the Bride Part II   

                                            overview  \
0  Led by Woody, Andy's toys live happily in his ...   
1  When siblings Judy and Peter discover an encha...   
2  A family wedding reignites the ancient feud be...   
3  Cheated on, mistreated and stepped on, the wom...   
4  Just when George Banks has recovered from his ...   

                         genres release_date  runtime  vote_average  \
0   [Animation, Comedy, Family]   1995-10-30     81.0           7.7   
1  [Adventure, Fantasy, Family]   1995-12-15    104.0           6.9   
2             [Romance, Comedy]   1995-12-22    101.0           6.5   
3      [Comedy, Drama, Romance]   1995-12-22    127.0           6.

In [11]:
print(movies_df.columns.tolist())

['id', 'title', 'overview', 'genres', 'release_date', 'runtime', 'vote_average', 'vote_count', 'popularity', 'genre_names', 'year', 'director', 'cast_names', 'cast_list']


## 4. Create FAISS Vector Store (RAG Core) 🧠

In [8]:
# Initialize embedding model
print("🔧 Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cuda' }
)
print("✅ Embedding model loaded!")

🔧 Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3957.33it/s]


✅ Embedding model loaded!


In [9]:
"""
Document(
    
Genres: Animation, Comedy, Family
Director: John Lasseter
Cast: Tom Hanks, Tim Allen, Don Rickles,....
Rating: 7.7/10
Runtime: 81 minutes

Plot:
Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene.""",

"""metadata={
        'title': 'Toy Story',
        'year': 1995,
        'genres': 'Animation, Comedy, Family',
        'director': 'John Lasseter',
        'rating': 7.7,
        'runtime': 81,
        'cast': 'Tom Hanks, Tim Allen, Don Rickles, ....',
        'id': 862
    }
)

"""

def create_documents(df):
    """Create LangChain documents from movie data"""
    print("📄 Creating documents for vector store...")
    
    documents = []
    
    for idx, row in df.iterrows():
        # OPTIMIZED: Shorter format
        content = f"""{row['title']} ({int(row['year'])})
Genre: {row['genre_names']}
Director: {row['director']}
Cast: {row['cast_names']}
Rating: {row['vote_average']}/10
\nPlot: {row['overview']}""".strip()
        
        metadata = {
            'title': row['title'],
            'year': int(row['year']),
            'genres': row['genre_names'],
            'director': row['director'],
            'rating': float(row['vote_average']),
            'runtime': int(row['runtime']) if pd.notna(row['runtime']) else 0,
            'cast': row['cast_names'],
            'id': int(row['id'])
        }
        
        doc = Document(page_content=content, metadata=metadata)
        documents.append(doc)
    
    print(f"✅ Created {len(documents)} documents")
    return documents

documents = create_documents(movies_df)

📄 Creating documents for vector store...
✅ Created 44503 documents


In [37]:
# Create FAISS vector store
print("🔨 Creating FAISS index... (this may take a few minutes)")
start_time = time.time()

vector_store = FAISS.from_documents(documents, embeddings)

duration = time.time() - start_time
print(f"✅ FAISS index created in {duration:.2f} seconds")

# Save vector store
vector_store.save_local("faiss_index")
print("💾 Vector store saved to disk!")

🔨 Creating FAISS index... (this may take a few minutes)
✅ FAISS index created in 124.43 seconds
💾 Vector store saved to disk!


In [10]:
#for loading the vector store later, you can use:
vector_store = FAISS.load_local(
    "faiss_index", 
    embeddings, 
    allow_dangerous_deserialization=True
)
print("✅ Vector store loaded from disk!")


✅ Vector store loaded from disk!


In [11]:
# Test vector store
print("🧪 Testing vector store...\n")

test_query = "movies about dreams and reality"
results = vector_store.similarity_search(test_query, k=3)

print(f"Query: '{test_query}'")
print(f"\nTop 3 results:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. {doc.metadata['title']} ({doc.metadata['year']})")
    print(f"   Genres: {doc.metadata['genres']}")
    print(f"   Rating: {doc.metadata['rating']}/10")

🧪 Testing vector store...

Query: 'movies about dreams and reality'

Top 3 results:

1. Have Dreams, Will Travel (2007)
   Genres: Drama, Romance
   Rating: 7.8/10

2. Dream (2008)
   Genres: Drama, Science Fiction
   Rating: 6.2/10

3. Dreamland (2007)
   Genres: Science Fiction, Horror, Mystery
   Rating: 3.2/10


## 5. Initialize Groq LLM 🤖

In [22]:
# Initialize Groq LLM
print("🤖 Initializing Groq LLM...")

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=MODEL_NAME,
    temperature=0.3,
    max_tokens=3000
)

# Test LLM
test_response = llm.invoke("Say ready")
print(f"✅ LLM initialized!")
print(f"Test response: {test_response.content}")

🤖 Initializing Groq LLM...
✅ LLM initialized!
Test response: Ready.


## 6. Watchlist Management System 📝

In [14]:
# Initialize watchlist
def init_watchlist():
    """Create watchlist file if doesn't exist"""
    if not os.path.exists(WATCHLIST_FILE):
        df = pd.DataFrame(columns=[
            'Title', 'Year', 'Genre', 'Status', 
            'Added_Date', 'Watched_Date', 'My_Rating', 'Notes'
        ])
        df.to_excel(WATCHLIST_FILE, index=False)
        print(f"✅ Created new watchlist: {WATCHLIST_FILE}")
    else:
        print(f"✅ Watchlist exists: {WATCHLIST_FILE}")

init_watchlist()

✅ Watchlist exists: my_watchlist.xlsx


In [ ]:
# Watchlist tool functions
class WatchlistManager:
    """Manages movie watchlist operations"""
    
    def __init__(self, movies_df, watchlist_file):
        self.movies_df = movies_df
        self.watchlist_file = watchlist_file
    
    def find_movie(self, movie_name: str):
        """Find movie in database (exact or fuzzy match)"""
        # Try exact match (case-insensitive)
        exact = self.movies_df[self.movies_df['title'].str.lower() == movie_name.lower()]
        if not exact.empty:
            return exact.iloc[0], None
        
        # Try partial match
        partial = self.movies_df[self.movies_df['title'].str.lower().str.contains(movie_name.lower(), na=False)]
        if not partial.empty:
            # Return best match + suggestions
            suggestions = partial.head(3)['title'].tolist()
            return None, suggestions
        
        return None, None
    
    def add_to_watchlist(self, movie_name: str) -> str:
        """Add movie to watchlist"""
        try:
            # Find movie
            movie, suggestions = self.find_movie(movie_name)
            
            if movie is None and suggestions:
                # Return suggestion for user to confirm
                return f"SUGGESTION|{suggestions[0]}|{'|'.join(suggestions[1:])}"
            
            if movie is None:
                return f"ERROR|Movie '{movie_name}' not found in database."
            
            # Load watchlist
            df = pd.read_excel(self.watchlist_file)
            
            # Check if already in watchlist
            if movie['title'] in df['Title'].values:
                return f"ERROR|'{movie['title']}' is already in your watchlist."
            
            # Add to watchlist
            new_row = {
                'Title': movie['title'],
                'Year': int(movie['year']),
                'Genre': movie['genre_names'],
                'Status': 'Want to Watch',
                'Added_Date': datetime.now().strftime('%Y-%m-%d'),
                'Watched_Date': None,
                'My_Rating': None,
                'Notes': None
            }
            
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            df.to_excel(self.watchlist_file, index=False)
            
            return f"SUCCESS|✅ Added '{movie['title']}' ({int(movie['year'])}) to your watchlist!"
            
        except Exception as e:
            return f"ERROR|Failed to add movie: {str(e)}"
    
    def mark_watched(self, movie_name: str, rating: str = "") -> str:
        """FIXED: Mark movie as watched with proper dtype handling for both date and rating"""
        try:
            df = pd.read_excel(self.watchlist_file)
            
            # CRITICAL FIX: Convert BOTH columns to object type
            df['Watched_Date'] = df['Watched_Date'].astype('object')
            df['My_Rating'] = df['My_Rating'].astype('object')  # ← NEW LINE!
            
            mask = df['Title'].str.lower() == movie_name.lower()
            
            if not mask.any():
                return f"'{movie_name}' not in watchlist."
            
            df.loc[mask, 'Status'] = 'Watched'
            df.loc[mask, 'Watched_Date'] = datetime.now().strftime('%Y-%m-%d')
            
            if rating:
                df.loc[mask, 'My_Rating'] = str(rating)
            
            # NEW: Retry logic for file locks
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    df.to_excel(self.watchlist_file, index=False)
                    break
                except PermissionError:
                    if attempt < max_retries - 1:
                        import time
                        time.sleep(0.5)
                    else:
                        return f"Error: Watchlist file is open. Please close it and try again."
            
            return f"✅ Marked '{df.loc[mask, 'Title'].values[0]}' as watched!"
        except Exception as e:
            return f"Error: {str(e)}"
        
    def cancel_movie(self, movie_name: str) -> str:
        """Cancel/remove movie from watchlist with retry logic"""
        try:
            df = pd.read_excel(self.watchlist_file)
            mask = df['Title'].str.lower() == movie_name.lower()
            
            if not mask.any():
                return f"'{movie_name}' not found in watchlist."
            
            df.loc[mask, 'Status'] = 'Cancelled'
            
            # NEW: Retry logic for file locks
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    df.to_excel(self.watchlist_file, index=False)
                    break
                except PermissionError:
                    if attempt < max_retries - 1:
                        import time
                        time.sleep(0.5)
                    else:
                        return f"Error: Watchlist file is open. Please close it and try again."
            
            return f"✅ Cancelled '{df.loc[mask, 'Title'].values[0]}' from watchlist."
        except Exception as e:
            return f"Error: {str(e)}"
    #////i want also to update in future to filter by rating and genre from watchlist
    def get_watchlist(self, status: str = "all") -> str:
        """Get movies from watchlist"""
        try:
            df = pd.read_excel(self.watchlist_file)
            
            if df.empty:
                return "SUCCESS|Watchlist is empty!"
            
            if status != "all":
                df = df[df['Status'].str.lower() == status.lower()]
            
            if df.empty:
                return f"SUCCESS|No movies with status '{status}'"
            
            # OPTIMIZED: Shorter format
            result = f"SUCCESS|📋 Watchlist ({len(df)} movies):\n\n"
            for idx, row in df.iterrows():
                result += f"{idx+1}. {row['Title']} ({row['Year']}) - {row['Status']}"
                if pd.notna(row['My_Rating']):
                    result += f" [{row['My_Rating']}]"
                result += "\n"
            
            return result
            
        except Exception as e:
            return f"ERROR|Failed: {str(e)}"
    
    def get_personalized_recommendations(self) -> str:
        """Recommend based on watch history"""
        try:
            df = pd.read_excel(self.watchlist_file)
            
            # Get watched movies with high ratings
            watched = df[df['Status'] == 'Watched']
            
            if watched.empty:
                return "ERROR|You haven't watched any movies yet! Watch some first to get personalized recommendations."
            
            # Get titles of high-rated movies
            high_rated = watched[watched['My_Rating'].str.contains('8|9|10', na=False)]
            
            if high_rated.empty:
                high_rated = watched  # Use all watched if no high ratings
            
            liked_titles = high_rated['Title'].tolist()
            
            return f"RECOMMEND|{','.join(liked_titles)}"
            
        except Exception as e:
            return f"ERROR|Failed to get recommendations: {str(e)}"

# Initialize watchlist manager
watchlist_mgr = WatchlistManager(movies_df, WATCHLIST_FILE)
print("✅ Watchlist manager initialized!")

✅ Watchlist manager initialized!


## 7. Memory Management System 💾

In [16]:
# Memory manager
class MemoryManager:
    """Manages conversation memory strictly keeping the last 4 messages to avoid rate limits"""
    
    def __init__(self, memory_file, backup_file, llm, max_recent=4, max_summary_words=500):
        self.memory_file = memory_file
        self.backup_file = backup_file
        self.llm = llm
        self.max_recent = max_recent # Only keep 4 messages (2 pairs)
        
        # Load or initialize memory
        if os.path.exists(memory_file):
            with open(memory_file, 'r') as f:
                self.memory = json.load(f)
            # Ensure we only load the last 4 if the existing file has more
            self.memory['recent_messages'] = self.memory.get('recent_messages', [])[-4:]
            print(f"📚 Loaded existing memory: {len(self.memory['recent_messages'])} recent messages")
        else:
            self.memory = {
                'recent_messages': [],
                'total_messages': 0
            }
            print("📝 Created new memory")
            
    def add_message(self, role: str, content: str):
        """Add message to memory and slice to keep only the last 4 messages (no summarization)"""
        # Truncate very long messages to save tokens
        if len(content) > 800:
            content = content[:800] + "..."
        
        message = {
            'role': role,
            'content': content,
            'timestamp': datetime.now().isoformat()
        }
        
        self.memory['recent_messages'].append(message)
        self.memory['total_messages'] += 1
        
        self._backup_message(message)
        
        # Keep ONLY the last 4 messages (2 pairs)
        # This completely removes the LLM call for summarization
        if len(self.memory['recent_messages']) > 4:
            self.memory['recent_messages'] = self.memory['recent_messages'][-4:]
        
        self._save()
    
    def get_context(self) -> str:
        """Get compact context for LLM (only the last 4 messages)"""
        context = ""
        
        if self.memory['recent_messages']:
            # No summary included, just the exact last 4 messages
            for msg in self.memory['recent_messages']:
                context += f"{msg['role']}: {msg['content'][:300]}\n"
        
        return context
    
    def _backup_message(self, message):
        """Backup to full history file"""
        if os.path.exists(self.backup_file):
            with open(self.backup_file, 'r') as f:
                backup = json.load(f)
        else:
            backup = {'messages': []}
        
        backup['messages'].append(message)
        
        with open(self.backup_file, 'w') as f:
            json.dump(backup, f, indent=2)
    
    def _save(self):
        """Save memory to disk"""
        with open(self.memory_file, 'w') as f:
            json.dump(self.memory, f, indent=2)
    
    def clear(self):
        """Clear memory"""
        self.memory = {
            'recent_messages': [],
            'total_messages': 0
        }
        self._save()
        print("🗑️ Memory cleared!")

# Initialize memory manager with max_recent set to 4
memory_mgr = MemoryManager(MEMORY_FILE, BACKUP_FILE, llm, max_recent=4)
print("✅ Memory manager initialized!")

📝 Created new memory
✅ Memory manager initialized!


## 8. Logging System 📊

In [17]:
# Simple logger
def log(message: str, level="INFO"):
    """Log message to file and console"""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_entry = f"[{timestamp}] [{level}] {message}"
    
    # Write to file
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(log_entry + '\n')
    
    # Print if error
    if level == "ERROR":
        print(f"❌ {log_entry}")

log("Session started", "INFO")
print("✅ Logging system ready!")

✅ Logging system ready!


## 9. Main Chatbot with Tools & Agent 🤖

In [18]:
# REPLACE THIS ENTIRE SECTION in your code:

def search_movies(query: str) -> str:
    """Search for movies"""
    try:
        results = vector_store.similarity_search(query, k=3)
        if not results:
            return "No movies found."
        
        output = "Found movies:\n\n"
        for i, doc in enumerate(results, 1):
            output += f"{i}. {doc.metadata['title']} ({doc.metadata['year']})\n"
            output += f"   {doc.metadata['genres']} | {doc.metadata['rating']}/10\n"
            plot = doc.page_content.split('Plot:')[1][:80] if 'Plot:' in doc.page_content else ''
            output += f"   {plot}...\n\n"
        return output
    except Exception as e:
        return f"Search error: {str(e)}"

def add_movie(movie_name: str) -> str:
    """Add movie to watchlist"""
    return watchlist_mgr.add_to_watchlist(movie_name)

def mark_watched(input_str: str) -> str:
    """Mark movie as watched. Input format: 'movie_name' or 'movie_name, rating'"""
    try:
        # Parse input: could be "Inception" or "Inception, 8/10"
        if ',' in input_str:
            parts = input_str.split(',')
            movie_name = parts[0].strip()
            rating = parts[1].strip() if len(parts) > 1 else ""
        else:
            movie_name = input_str.strip()
            rating = ""
        
        return watchlist_mgr.mark_watched(movie_name, rating)
    except Exception as e:
        return f"Error: {str(e)}"

def cancel_movie(movie_name: str) -> str:
    """Cancel/remove movie from watchlist"""
    return watchlist_mgr.cancel_movie(movie_name)

def view_watchlist(input_str: str = "") -> str:
    """View watchlist. Input is ignored, just pass empty string."""
    return watchlist_mgr.get_watchlist()

def get_recommendations(input_str: str = "") -> str:
    """Get personalized recommendations. Input is ignored."""
    try:
        df = pd.read_excel(WATCHLIST_FILE)
        watched = df[df['Status'] == 'Watched']

        if watched.empty:
            return "Watch some movies first, then I can recommend similar ones!"

        high_rated = watched[watched['My_Rating'].notna()]
        if high_rated.empty:
            high_rated = watched

        liked_titles = high_rated['Title'].tail(2).tolist()
        query = f"movies similar to {' and '.join(liked_titles)}"
        results = vector_store.similarity_search(query, k=5)

        all_watchlist = df['Title'].tolist()
        recommendations = [r for r in results if r.metadata['title'] not in all_watchlist]

        if not recommendations:
            return "No new recommendations found. Try watching more movies!"

        output = f"Based on what you've watched, try:\n\n"
        for i, doc in enumerate(recommendations[:3], 1):
            output += f"{i}. {doc.metadata['title']} ({doc.metadata['year']})\n"
            output += f"   {doc.metadata['genres']} | {doc.metadata['rating']}/10\n\n"

        return output
    except Exception as e:
        return f"Error: {str(e)}"

tools = [
    Tool(name="search_movies", func=search_movies, description="Search movies. Input: query (e.g. 'sci-fi movies')"),
    Tool(name="add_to_watchlist", func=add_movie, description="Add movie. Input: movie title"),
    Tool(name="mark_watched", func=mark_watched, description="Mark watched. Input: 'movie title' or 'movie title, rating'"),
    Tool(name="cancel_movie", func=cancel_movie, description="Remove movie. Input: movie title"),
    Tool(name="view_watchlist", func=view_watchlist, description="View watchlist. Input: empty string or anything"),
    Tool(name="get_recommendations", func=get_recommendations, description="Get recommendations. Input: empty string or anything")
]

In [26]:
# REPLACE THIS SECTION:

prompt_template = """You are a helpful movie assistant.

Tools available:
{tools}

Always use this exact format:

Question: the user's question
Thought: I should [action to take]
Action: [tool name from: {tool_names}]
Action Input: [the input]
Observation: [tool result]
Thought: I now know the answer
Final Answer: [your response to user]

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(prompt_template)

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors="Check your format. Use: Thought, Action, Action Input.",
    max_iterations=3
)

In [20]:
# Main chatbot function
def chat(user_input: str) -> str:
    """Main chat function with memory and performance monitoring"""
    start_time = time.time()
    
    try:
        # Log user input
        log(f"User: {user_input}")
        
        # Add to memory
        memory_mgr.add_message("user", user_input)
        
        # Get memory context
        context = memory_mgr.get_context()
        
        # Prepare input with context
        full_input = f"""{context}

Current question: {user_input}"""
        
        # Run agent
        response = agent_executor.invoke({"input": full_input})
        answer = response['output']
        
        # Handle special responses (suggestions, errors)
        if "SUGGESTION|" in answer:
            parts = answer.split("|")
            if len(parts) >= 2:
                suggestion = parts[1]
                answer = f"I couldn't find that exact movie. Did you mean '{suggestion}'? (Please reply Yes or No)"
        elif "ERROR|" in answer:
            answer = answer.split("|")[1] if "|" in answer else answer
        elif "SUCCESS|" in answer:
            answer = answer.split("|")[1] if "|" in answer else answer
        
        # Add to memory
        memory_mgr.add_message("assistant", answer)
        
        # Log response
        log(f"Assistant: {answer[:100]}...")
        
        # Performance monitoring
        duration = time.time() - start_time
        print(f"\n⏱️ Response time: {duration:.2f} seconds")
        log(f"Response time: {duration:.2f}s")
        
        return answer
        
    except Exception as e:
        error_msg = f"Sorry, I encountered an error: {str(e)}"
        log(f"Error: {str(e)}", "ERROR")
        
        # Graceful degradation: try simple response
        try:
            simple_response = llm.invoke(f"Answer briefly: {user_input}").content
            return simple_response
        except:
            return error_msg

print("✅ Chatbot ready!")

✅ Chatbot ready!


## 11. Interactive Chat Interface 💬

In [24]:
# Interactive chat loop
def interactive_chat():
    """Interactive chat session"""
    print("\n🎬 Movie Recommendation Assistant")
    print("="*50)
    print("Type 'quit' or 'exit' to end conversation")
    print("Type 'clear' to clear memory")
    print("Type 'watchlist' to see your watchlist")
    print("="*50 + "\n")
    
    while True:
        user_input = input("\n You: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\n👋 Goodbye! Happy watching!")
            break
        
        if user_input.lower() == 'clear':
            memory_mgr.clear()
            print("🗑️ Memory cleared!")
            continue
        
        if user_input.lower() == 'watchlist':
            result = watchlist_mgr.get_watchlist()
            print(f"\n🤖 Bot:\n{result.split('|')[1] if '|' in result else result}")
            continue
        
        # Get response
        response = chat(user_input)
        print(f"\n🤖 Bot:\n{response}")

# Uncomment to start interactive chat
interactive_chat()


🎬 Movie Recommendation Assistant
Type 'quit' or 'exit' to end conversation
Type 'clear' to clear memory
Type 'watchlist' to see your watchlist



> Entering new AgentExecutor chain...
Could not parse LLM output: `The movie **Inception** has been removed from your watchlist. Let me know if there's anything else I can help with!`
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE Check your format. Use: Thought, Action, Action Input.Question: cancel inception
Thought: I should remove the movie "Inception" from the user's watchlist.
Action: cancel_movie
Action Input: Inception✅ Cancelled 'Inception' from watchlist.Question: cancel inception
Thought: I have cancelled "Inception" from the user's watchlist.
Action: cancel_movie
Action Input: Inception✅ Cancelled 'Inception' from watchlist.

> Finished chain.

⏱️ Response time: 1.49 seconds

🤖 Bot:
Agent stopped due to iteration limit or time limit.


> Entering new AgentExecutor chain..

## 12. Quick Test - Single Query 🚀

In [27]:
# Quick single test
test_query = "What are some mind-bending sci-fi movies?"

print(f"❓ Question: {test_query}\n")
response = chat(test_query)
print(f"\n💬 Answer:\n{response}")

❓ Question: What are some mind-bending sci-fi movies?



> Entering new AgentExecutor chain...
Question: What are some mind-bending sci-fi movies?
Thought: I should retrieve a list of mind-bending sci-fi movies using the recommendations tool.
Action: get_recommendations
Action Input: Based on what you've watched, try:

1. The Godfather Trilogy: 1972-1990 (1992)
   Unknown | 8.8/10

2. The Godfather: Part II (1974)
   Drama, Crime | 8.3/10

3. The Godfather: Part III (1990)
   Crime, Drama, Thriller | 7.1/10

Question: What are some mind-bending sci‑fi movies?  
Thought: I should look up movies that are commonly described as mind‑bending sci‑fi.  
Action: search_movies  
Action Input: mind-bending sci-fi movies  Found movies:

1. MindGamers (2015)
   Action, Science Fiction, Thriller | 6.0/10
    A group of young bio-engineers discover they can use quantum physics to transfe...

2. Mindwarp (1992)
   Horror, Science Fiction, Thriller | 5.5/10
    Revolting mutants hunt human outcasts an

In [29]:
# Quick single test
test_query = "Add coco to my watchlist"

print(f"❓ Question: {test_query}\n")
response = chat(test_query)
print(f"\n💬 Answer:\n{response}")

❓ Question: Add coco to my watchlist



> Entering new AgentExecutor chain...
Could not parse LLM output: ``
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE Check your format. Use: Thought, Action, Action Input.Question: Add coco to my watchlist
Thought: I should add the movie "Coco" to the user's watchlist using the add_to_watchlist tool.
Action: add_to_watchlist
Action Input: CocoSUCCESS|✅ Added 'Coco' (2009) to your watchlist!Question: Add coco to my watchlist
Thought: I should add the movie "Coco" to the user's watchlist using the add_to_watchlist tool.
Action: add_to_watchlist
Action Input: CocoERROR|'Coco' is already in your watchlist.

> Finished chain.

⏱️ Response time: 2.23 seconds

💬 Answer:
Agent stopped due to iteration limit or time limit.


In [30]:
# Quick single test
test_query = "Recommend me some movies similar to Inception and Interstellar. and i watched coco and i liked it rate it 9 the movie was amazing and the twist at the end was mind blowing"

print(f"❓ Question: {test_query}\n")
response = chat(test_query)
print(f"\n💬 Answer:\n{response}")

❓ Question: Recommend me some movies similar to Inception and Interstellar. and i watched coco and i liked it rate it 9 the movie was amazing and the twist at the end was mind blowing



> Entering new AgentExecutor chain...
Question: Recommend me some movies similar to Inception and Interstellar. and i watched coco and i liked it rate it 9 the movie was amazing and the twist at the end was mind blowing  
Thought: I should record that the user watched **Coco** and gave it a rating of 9.  
Action: mark_watched  
Action Input: Coco, 9  ✅ Marked 'Coco' as watched!Question: Recommend me some movies similar to Inception and Interstellar. and i watched coco and i liked it rate it 9 the movie was amazing and the twist at the end was mind blowing  
Thought: I should record that the user watched **Coco** and gave it a rating of 9.  
Action: mark_watched  
Action Input: Coco, 9  ✅ Marked 'Coco' as watched!Could not parse LLM output: `Here are some movies you might enjoy based on your love for **

## 🎉 Setup Complete!

### How to Use:

1. **Run test suite:** Uncomment and run `run_tests()` cell
2. **Interactive chat:** Uncomment and run `interactive_chat()` cell
3. **Single query:** Use the quick test cell

### Features:
- ✅ RAG with FAISS (semantic search on 40k+ movies)
- ✅ LangChain ReAct agent with 6 tools
- ✅ Conversation memory with auto-summarization
- ✅ Interactive validation (asks user when uncertain)
- ✅ Performance monitoring (response times)
- ✅ Graceful degradation (fallback when errors)
- ✅ Logging system (tracks all interactions)
- ✅ Watchlist management (add/watch/cancel)
- ✅ Personalized recommendations

### Next Steps:
1. Test with different queries
2. Add more movies to watchlist
3. Check memory summarization after 10+ messages
4. Review logs for debugging

---
**Good luck with your project! 🚀**